# How Toxic Is X (Twitter)? Analysis

In this notebook we take the tweets we collected with `collect_x.py`, clean them, score them with three methods, and answer our three questions:

1. How toxic is each topic, and does it match our group's guess?
2. What kind of toxicity is it?
3. How accurate is each method? We check this against our own hand labels.

Figures are saved to `figures/` with the same numbers as in the report (`fig1_...` = Figure 1, ..., `fig5_...` = Figure 5). The tables in section 7 are the report's appendix.

The scoring step is slow, so we save the scores to `data/scored.csv` and reuse them next time.

> This notebook shows tweet text in its outputs. Before committing, clear them with
> `jupyter nbconvert --clear-output --inplace toxicity_analysis.ipynb`

## 0. Setup

In [ ]:
import html
import json
import re
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

DATA = Path("data")
RAW = DATA / "raw" / "combined.csv"
CLEAN = DATA / "clean.csv"
SCORED = DATA / "scored.csv"
SCORE_INFO = DATA / "scored_info.json"
FIG = Path("figures")
FIG.mkdir(exist_ok=True)
BADWORDS = Path("resources/ldnoobw_en.txt")

THRESHOLD = 0.5
THRESHOLDS = [0.2, 0.5, 0.8]   # for section 4.2 and 6.3
SEED = 42

TOPIC_ORDER = ["us_politics", "kpop", "movies", "tech", "pets", "food"]
TOPIC_NAME = {"us_politics": "US Politics", "kpop": "K-pop", "movies": "Movies",
              "tech": "Tech", "pets": "Pets", "food": "Food"}
GUESS = {"us_politics": "High", "kpop": "High", "movies": "Mid",
         "tech": "Mid", "pets": "Low", "food": "Low"}

pd.set_option("display.max_colwidth", 100)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## 1. Load the data

We start from `combined.csv`. It has all posts and replies from the 6 topics in one table, and the `type` column tells them apart.

In [ ]:
raw = pd.read_csv(RAW, dtype={"tweet_id": str, "parent_id": str})
raw["text"] = raw["text"].fillna("")
raw["parent_text"] = raw["parent_text"].fillna("")
print(f"{len(raw):,} tweets")
raw[["topic", "hashtag", "type", "text"]].head()

In [ ]:
raw.pivot_table(index="topic", columns="type", values="tweet_id", aggfunc="count").reindex(TOPIC_ORDER)

## 2. Clean

Many posts are ads or shop links, not people talking. Replies also start with @names that are not part of the message. Our plan:

1. Remove **promo** tweets: a link **and** 3 or more hashtags. We check this before cleaning, while the links are still there.
2. Clean the text: decode `&amp;`, remove the @names at the start of replies, remove links.
3. Remove the same text repeated in one topic (copy-paste fan campaigns, bots).
4. Remove texts shorter than 3 words ("lol", "W"). They are too short to judge.

We keep two versions of the text. `text_clean` has no @names or links, and we use it for the word list and to show tweets. `text_model` replaces them with `@user` and `http`, because that's how Twitter-RoBERTa was trained.

In [ ]:
URL = re.compile(r"https?://\S+")
LEAD_MENTIONS = re.compile(r"^(?:\s*@\w+)+")
MENTION = re.compile(r"@\w+")


def clean_text(t):
    t = html.unescape(t)
    t = LEAD_MENTIONS.sub("", t)
    t = URL.sub("", t)
    t = MENTION.sub("", t)
    return " ".join(t.split())


def model_text(t):
    t = html.unescape(t)
    t = LEAD_MENTIONS.sub("", t)
    t = URL.sub("http", t)
    t = MENTION.sub("@user", t)
    return " ".join(t.split())


df = raw.copy()
df["is_promo"] = df["text"].str.contains(URL) & (df["text"].str.count(r"#\w+") >= 3)
df["text_clean"] = df["text"].map(clean_text)
df["text_model"] = df["text"].map(model_text)
df["parent_clean"] = df["parent_text"].map(clean_text)
df[["type", "is_promo", "text", "text_clean"]].head()

Now we apply the four steps and count how many tweets are left after each one.

In [ ]:
steps = {}
steps["1. collected"] = df.groupby(["topic", "type"]).size()

# step 1: remove promo
df = df[~df["is_promo"]].copy()
steps["2. no promo"] = df.groupby(["topic", "type"]).size()

# step 3: remove the same text in the same topic and type (ignore upper/lower case)
df["key"] = df["text_clean"].str.lower()
df = df.drop_duplicates(["topic", "type", "key"])
df = df.drop(columns="key")
steps["3. no duplicates"] = df.groupby(["topic", "type"]).size()

# step 4: at least 3 words
n_words = df["text_clean"].str.split().str.len()
df = df[n_words >= 3]
steps["4. 3+ words"] = df.groupby(["topic", "type"]).size()

clean = df.reset_index(drop=True)
assert clean["tweet_id"].is_unique
save_cols = ["topic", "level", "hashtag", "type", "tweet_id", "parent_id", "created_at",
             "text_clean", "text_model", "parent_clean"]
clean[save_cols].to_csv(CLEAN, index=False, encoding="utf-8")

print(f"Kept {len(clean):,} of {len(raw):,} tweets")
table = pd.DataFrame(steps).unstack("type").fillna(0).astype(int)
table = table.reindex(TOPIC_ORDER).rename(index=TOPIC_NAME)
table

The promo filter removes most **posts** in pets, food and K-pop, which were mostly ads. Replies are almost untouched, because people write them, not shops.

## 3. Score

We score every tweet with three methods:

| Method | Idea | Output |
|---|---|---|
| Bad-word list | Is any word from the LDNOOBW list in the tweet? No learning | number of bad words |
| Detoxify (unbiased) | AI model trained on online comments | 7 scores from 0 to 1 (see below) |
| Twitter-RoBERTa | AI model trained on tweets | 1 score from 0 to 1 (offensive) |

A tweet counts as toxic when the score is 0.5 or more (or, for the list, when it has at least one bad word).

**The 7 Detoxify scores.** Each score is separate, from 0 to 1, so one tweet can be high on several at once (for example, both insult and obscene).

| Score | What it means | How we use it |
|---|---|---|
| `toxicity` | Rude or hateful in general | **Toxic or not** (>= 0.5). Used for Question 1 and 3 |
| `severe_toxicity` | Very hateful or aggressive | Not used. It says *how* toxic, not *what kind* |
| `obscene` | Swearing, vulgar words | **Type of toxicity** (Question 2) |
| `insult` | Insulting a person | **Type of toxicity** (Question 2) |
| `threat` | Wanting to hurt someone | **Type of toxicity** (Question 2) |
| `identity_attack` | Attacking a group (race, religion, gender, ...) | **Type of toxicity** (Question 2) |
| `sexual_explicit` | Sexual content | **Type of toxicity** (Question 2) |

So we use `toxicity` to decide **whether** a tweet is toxic, and then, only for toxic tweets, the highest of the 5 type scores to decide **what kind** it is. (The model also gives 9 more scores for which group is mentioned, like `muslim` or `female`. We don't need those, so we drop them.)

**Twitter-RoBERTa** gives 2 scores, `non-offensive` and `offensive`, which add up to 1. We use only `offensive`.

### 3.1 Bad-word list

We use the English list from **LDNOOBW** (*List of Dirty, Naughty, Obscene, and Otherwise Bad Words*, https://github.com/LDNOOBW/List-of-Dirty-Naughty-Obscene-and-Otherwise-Bad-Words), saved as `resources/ldnoobw_en.txt` (403 words and phrases). It is one of the most widely used bad-word lists on GitHub (about 3.5k stars). We also tested two other lists, [badwordslist](https://code.google.com/archive/p/badwordslist/) (Google Code) and [google-profanity-words](https://github.com/coffee-and-fun/google-profanity-words) (coffee-and-fun). They found a few more toxic tweets, but they also flagged harmless words, such as food ("turkey breasts") and casual words ("hell yeah"). All three lists still missed more than half of the tweets the AI models call toxic, so choosing another list would not change our answer to Question 3.

We match whole words, not parts of words. So "class" doesn't match "ass".

In [ ]:
from nltk.tokenize import TweetTokenizer

words = []
for line in BADWORDS.read_text(encoding="utf-8").splitlines():
    if line.strip():
        words.append(line.strip().lower())

BW_SINGLE = set(w for w in words if " " not in w)
BW_PHRASES = [w for w in words if " " in w]
BW_PHRASES.sort(key=len, reverse=True)   # longest first
BW_PHRASE_RE = re.compile(r"\b(" + "|".join(re.escape(p) for p in BW_PHRASES) + r")\b")
tokenizer = TweetTokenizer(preserve_case=False, reduce_len=True)


def badword_hits(text):
    t = text.lower().replace("#", "")
    hits = []
    for w in tokenizer.tokenize(t):
        if w in BW_SINGLE:
            hits.append(w)
    hits += BW_PHRASE_RE.findall(t)
    return hits


print(f"{len(BW_SINGLE)} words and {len(BW_PHRASES)} phrases in the list")
for s in ["This game is fucking awesome!", "You are so stupid, uninstall the game", "first class ticket"]:
    print(s, "->", badword_hits(s))

### 3.2 AI models

We load the Detoxify `unbiased` model weights directly with `transformers` (the `detoxify` package doesn't install cleanly with our versions). Detoxify gives several labels at once, so we use **sigmoid**. RoBERTa picks one of two classes, so we use **softmax**.

If `scored.csv` already has exactly the same tweets as `clean.csv`, we just load it. If the cleaning changed, we score again.

In [ ]:
DETOX_ID = "unitary/unbiased-toxic-roberta"
ROBERTA_ID = "cardiffnlp/twitter-roberta-base-offensive"
DETOX_COLS = ["toxicity", "severe_toxicity", "obscene", "identity_attack",
              "insult", "threat", "sexual_explicit"]


def run_model(model_id, texts, activation, batch_size=32):
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSequenceClassification.from_pretrained(model_id)
    model.eval()
    labels = [model.config.id2label[i] for i in range(model.config.num_labels)]

    start = time.time()
    results = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            enc = tok(batch, padding=True, truncation=True, max_length=128, return_tensors="pt")
            logits = model(**enc).logits
            if activation == "sigmoid":
                probs = torch.sigmoid(logits)
            else:
                probs = torch.softmax(logits, dim=-1)
            results.append(probs.numpy())
    print(f"{model_id}: {len(texts):,} tweets in {time.time() - start:.0f}s")
    return pd.DataFrame(np.vstack(results), columns=labels)


def score_all(clean):
    s = clean.copy()
    s["bw_words"] = ["|".join(badword_hits(t)) for t in s["text_clean"]]
    s["bw_count"] = [len(x.split("|")) if x else 0 for x in s["bw_words"]]
    texts = s["text_model"].tolist()

    detox = run_model(DETOX_ID, texts, "sigmoid")
    for c in DETOX_COLS:   # we don't need the identity-group labels
        s["detox_" + c] = detox[c].to_numpy()

    roberta = run_model(ROBERTA_ID, texts, "softmax")
    s["roberta_offensive"] = roberta["offensive"].to_numpy()
    return s


clean = pd.read_csv(CLEAN, dtype={"tweet_id": str, "parent_id": str})
clean = clean.fillna({"parent_clean": "", "parent_id": ""})

# scoring is slow, so reuse scored.csv if it has the same tweets as clean.csv
scored = None
if SCORED.exists():
    old = pd.read_csv(SCORED, dtype={"tweet_id": str, "parent_id": str})
    if len(old) == len(clean) and set(old["tweet_id"]) == set(clean["tweet_id"]):
        scored = old
        print(f"Loaded saved scores ({len(scored):,} tweets). Delete {SCORED} to score again.")

if scored is None:
    scored = score_all(clean)
    scored.to_csv(SCORED, index=False, encoding="utf-8")
    import torch
    import transformers
    info = {"date": datetime.now().isoformat(timespec="seconds"),
            "detoxify_model": DETOX_ID, "roberta_model": ROBERTA_ID,
            "transformers": transformers.__version__, "torch": torch.__version__,
            "n_tweets": len(scored)}
    SCORE_INFO.write_text(json.dumps(info, indent=2))

scored = scored.fillna({"bw_words": "", "parent_id": "", "parent_clean": ""})
scored["detox_toxic"] = (scored["detox_toxicity"] >= THRESHOLD).astype(int)
scored["roberta_toxic"] = (scored["roberta_offensive"] >= THRESHOLD).astype(int)
scored["bw_toxic"] = (scored["bw_count"] > 0).astype(int)
scored[["topic", "type", "bw_count", "detox_toxicity", "detox_insult", "roberta_offensive"]].head()

Before we trust the scores, we do a quick sanity check: is the scoring working the right way round? We take two tweets from our own data whose answer is obvious to any person: one clearly toxic (a personal attack) and one clearly friendly (a kind message to someone whose pet died). The toxic one should score high and the friendly one low. If not, there is a bug, for example reading the wrong output column.

In [ ]:
for name, needle in [("toxic example", "YOU ARE THE DUMBEST"), ("friendly example", "RIP SWEET ANGEL")]:
    r = scored[scored["text_clean"].str.contains(needle, case=False, regex=False)].iloc[0]
    print(f"{name:16}: Detoxify {r['detox_toxicity']:.2f}, RoBERTa {r['roberta_offensive']:.2f}, bad words {r['bw_count']}")

## 4. Question 1: How toxic is each topic, and does it match our guess?

**Replies are our main result**, because that's where people talk to each other. We answer in three steps:

1. **% toxic per topic** for each method (4.1).
2. **Threshold check** (4.2). Does the topic order change if we use another threshold than 0.5?
3. **Our guess vs. the data** (4.3). Sort the topics into High, Mid and Low like our vote, and count how many match.

### 4.1 % toxic per topic

The three methods side by side: the % of toxic tweets per topic, with the number of toxic tweets and all tweets in brackets. Small groups (like Pets replies, 53 tweets) can change a lot with one tweet.

Each method has its own rule for "toxic":

| Method | A tweet is toxic when |
|---|---|
| Detoxify | `toxicity` score >= 0.5 |
| RoBERTa | `offensive` score >= 0.5 |
| Word list | it has at least 1 word from the list |

0.5 is the usual default for these models.

In [ ]:
methods = {"Detoxify": "detox_toxic", "RoBERTa": "roberta_toxic", "Word list": "bw_toxic"}


def method_table(kind):
    g = scored[scored["type"] == kind]
    table = pd.DataFrame(index=[TOPIC_NAME[t] for t in TOPIC_ORDER])
    table.index.name = "topic"
    for name, col in methods.items():
        cells = []
        for t in TOPIC_ORDER:
            n = (g["topic"] == t).sum()
            k = g.loc[g["topic"] == t, col].sum()
            cells.append(f"{100 * k / n:.1f}% ({k}/{n})")
        table[name] = cells
    return table


totals = [f"{name} {scored[col].sum()}" for name, col in methods.items()]
print("Toxic tweets in total: " + ", ".join(totals) + f" (of {len(scored):,})")
print()
print("Replies: % toxic (toxic/all)")
display(method_table("reply"))
print("Posts: % toxic (toxic/all)")
display(method_table("post"))

In [ ]:
colors = {"Detoxify": "#c0392b", "RoBERTa": "#2980b9", "Word list": "#7f8c8d"}
x = np.arange(6)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)   # same y-axis for both
for ax, kind, title in [(axes[0], "reply", "Replies"), (axes[1], "post", "Posts")]:
    g = scored[scored["type"] == kind]
    pct = g.groupby("topic")[list(methods.values())].mean().reindex(TOPIC_ORDER) * 100
    n = g.groupby("topic").size()
    for i, name in enumerate(colors):
        ax.bar(x + (i - 1) * 0.27, pct[methods[name]], 0.27, label=name, color=colors[name])
    ax.set_xticks(x, [f"{TOPIC_NAME[t]}\nn={n[t]}" for t in TOPIC_ORDER])
    ax.set_title(f"{title}: % toxic by method")
axes[0].set_ylabel("% toxic tweets")
axes[0].legend(frameon=False)
plt.tight_layout()
plt.savefig(FIG / "fig1_toxic_by_topic.png", bbox_inches="tight")
plt.show()

**What we see**

- **All three methods put US Politics first**, for replies and for posts. For replies: Detoxify 21.4%, RoBERTa 32.4%, word list 11.0%.
- **Detoxify and RoBERTa give the same order for replies:** US Politics > K-pop > Pets > Movies > Food > Tech. RoBERTa flags more tweets overall (160 vs 122).
- **The word list finds almost nothing outside politics:** 0–2 replies per topic (0–1.9%). So its order for the other 5 topics depends on 1 or 2 tweets.
- **Posts are closer together.** Apart from US Politics (20–24% with the AI models), all topics are between 2% and 5%, so their order can change with one tweet.

### 4.2 Does the order depend on the threshold?

We count toxic tweets per topic with Detoxify and RoBERTa at 3 thresholds: 0.2, 0.5 and 0.8. If the topic order stays about the same, the result doesn't depend on our choice of 0.5. The word list has no score, only a word count, so a threshold doesn't apply to it. The `match` column (how many topics match our guess) is explained in 4.3.

In [ ]:
def to_levels(pct):
    # top 2 topics = High, middle 2 = Mid, bottom 2 = Low (same as our vote)
    order = pct.sort_values(ascending=False, kind="stable").index
    return pd.Series(["High", "High", "Mid", "Mid", "Low", "Low"], index=order)


def n_match(levels):
    count = 0
    for t in TOPIC_ORDER:
        if levels[t] == GUESS[t]:
            count += 1
    return count


SCORES = {"Detoxify": "detox_toxicity", "RoBERTa": "roberta_offensive"}
rows = []
for kind in ["reply", "post"]:
    g = scored[scored["type"] == kind]
    for name, col in SCORES.items():
        for th in THRESHOLDS:
            pct = (g[col] >= th).groupby(g["topic"]).mean().reindex(TOPIC_ORDER) * 100
            row = {"type": kind, "method": name, "threshold": th}
            for t in TOPIC_ORDER:
                row[TOPIC_NAME[t]] = round(pct[t], 1)
            row["match"] = f"{n_match(to_levels(pct))}/6"
            rows.append(row)

print("% toxic per topic at each threshold")
pd.DataFrame(rows).set_index(["type", "method", "threshold"])

**How the scores are spread.** Detoxify and RoBERTa give very different scores, so the same threshold may not suit both.

In [ ]:
MODEL_COLOR = {"Detoxify": "#c0392b", "RoBERTa": "#2980b9"}

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
for ax, name in zip(axes, SCORES):
    col = SCORES[name]
    ax.hist(scored[col], bins=np.linspace(0, 1, 21), color=MODEL_COLOR[name], edgecolor="white")
    for th in THRESHOLDS:
        n_above = (scored[col] >= th).sum()
        ax.axvline(th, ls="--", lw=1, color="#7f8c8d")
        ax.text(th, 1.02, f"{th} ({n_above})", transform=ax.get_xaxis_transform(),
                ha="center", fontsize=8, color="#555555")
    ax.set_yscale("log")
    ax.set_xlabel(f"{name} score")
    ax.set_title(f"{name}: how the scores are spread", pad=16)
axes[0].set_ylabel("tweets (log scale)")
plt.tight_layout()
plt.savefig(FIG / "fig2_score_spread.png", bbox_inches="tight")
plt.show()
print("Numbers above the dashed lines = tweets with a score at or above that threshold")

**What this shows**

- **Detoxify is almost yes-or-no.** Most tweets score near 0 or near 1, so moving the threshold from 0.2 to 0.8 only changes the number of toxic tweets from 165 to 89. **RoBERTa is spread out:** 450 tweets at 0.2 but only 50 at 0.8. So the threshold matters much more for RoBERTa.
- **So we keep 0.5 for both models**, the usual default. Section 6.3 checks with our hand labels which threshold is actually right.

### 4.3 Our guess vs. the data

Our vote put 2 topics in each level, so we do the same with the data: sort the 6 topics by % toxic, then top 2 = **High**, middle 2 = **Mid**, bottom 2 = **Low**. The levels are relative because there is no standard cut-off (like "10% = High"), and the methods give very different %. Then we count how many topics match our guess.

In [ ]:
def level_table(kind):
    # % toxic and the data level for each topic (models >= 0.5, list >= 1 bad word)
    g = scored[scored["type"] == kind]
    table = pd.DataFrame(index=TOPIC_ORDER)
    table["our guess"] = [GUESS[t] for t in TOPIC_ORDER]
    match_row = [""]
    for name, col in methods.items():
        pct = g.groupby("topic")[col].mean().reindex(TOPIC_ORDER) * 100
        levels = to_levels(pct)
        cells = []
        for t in TOPIC_ORDER:
            s = f"{pct[t]:.1f}% {levels[t]}"
            if levels[t] == GUESS[t]:
                s += " *"
            cells.append(s)
        table[name] = cells
        match_row.append(f"{n_match(levels)}/6")
    table.loc["matches"] = match_row
    return table.rename(index=TOPIC_NAME)


print("Replies (* = same level as our guess)")
display(level_table("reply"))
print("Posts")
display(level_table("post"))

g = scored.groupby("type")["detox_toxic"].mean() * 100
print(f"All topics together (Detoxify): replies {g['reply']:.1f}% toxic, posts {g['post']:.1f}% toxic")

In [ ]:
# slope chart, replies: our guess (left) vs the data (right)
# dashed lines = High / Mid / Low cut, red = topic is in a different level than we guessed
reply = scored[scored["type"] == "reply"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4.3), sharey=True)
for ax, name in zip(axes, methods):
    pct = reply.groupby("topic")[methods[name]].mean().reindex(TOPIC_ORDER) * 100
    levels = to_levels(pct)
    for t in TOPIC_ORDER:
        y_left = TOPIC_ORDER.index(t)        # TOPIC_ORDER is already High, High, Mid, Mid, Low, Low
        y_right = list(levels.index).index(t)
        if levels[t] == GUESS[t]:
            color, lw = "#7f8c8d", 1.5
        else:
            color, lw = "#c0392b", 2.5
        ax.plot([0, 1], [y_left, y_right], "-o", lw=lw, color=color, zorder=3)
        ax.text(-0.06, y_left, TOPIC_NAME[t], ha="right", va="center", fontsize=9)
        ax.text(1.06, y_right, f"{TOPIC_NAME[t]} {pct[t]:.1f}%", ha="left", va="center", fontsize=9,
                color="black" if levels[t] == GUESS[t] else "#c0392b")
    ax.axhline(1.5, ls="--", lw=0.8, color="#bdc3c7", zorder=1)
    ax.axhline(3.5, ls="--", lw=0.8, color="#bdc3c7", zorder=1)
    for y, lvl in [(0.5, "High"), (2.5, "Mid"), (4.5, "Low")]:
        ax.text(0.5, y, lvl, ha="center", va="center", fontsize=8, color="#95a5a6", style="italic",
                bbox={"facecolor": "white", "edgecolor": "none", "pad": 1}, zorder=2)
    ax.set_xlim(-0.75, 1.85)
    ax.set_xticks([0, 1], ["Our guess", "Data"])
    ax.set_title(f"{name}: {n_match(levels)}/6 match")
    ax.tick_params(left=False)
    ax.spines["left"].set_visible(False)
    ax.spines["bottom"].set_visible(False)
axes[0].set_ylim(5.6, -0.6)
axes[0].set_yticks([])
fig.suptitle("Our guess vs. the data (replies). Red = the topic is in a different level than we guessed", fontsize=11)
plt.tight_layout()
plt.savefig(FIG / "fig3_guess_vs_data.png", bbox_inches="tight")
plt.show()

**What this shows**

- **US Politics is #1 in every case:** both models, all 3 thresholds, replies and posts.
- **Replies are stable for Detoxify:** 4/6 at 0.2, 0.5 and 0.8. At 0.5, both models miss the same 2 topics: Tech is less toxic than we guessed (Low, not Mid), and Pets is more toxic (Mid, not Low).
- **RoBERTa changes with the threshold:** 3/6 at 0.2, 4/6 at 0.5, 2/6 at 0.8. At 0.2 it is too loose: it flags 68% of US Politics replies and about a third of the replies in every other topic except Tech. At 0.8 it is too strict: Movies, Tech, Pets and Food have only 0 or 1 toxic replies each, so their order depends on single tweets.
- **Posts are not stable (1–4/6).** At 0.5, apart from US Politics, all topics are between 2% and 6%, so one tweet can change the order. For example, Detoxify at 0.5 has Movies (5.3%, 7 of 133) just above K-pop (5.2%, 6 of 115).
- **The word list matches only 1/6 for replies.** Outside politics it finds 0–2 bad-word replies per topic, so its ranking isn't reliable.

**Answer to Question 1**

- **US Politics is the most toxic topic.** 21.4% of its replies are toxic (37 of 173, Detoxify), and it is #1 for every method and threshold.
- **K-pop is second** (11.8%, 12 of 102). The other four topics are all below 6%.
- **Tech is the least toxic** (0.6%, 1 of 169).
- **Our guess was mostly right:** 4/6 topics for replies, with both models at 0.5, and with Detoxify at every threshold (RoBERTa drops to 3/6 at 0.2 and 2/6 at 0.8). We guessed too high for Tech and too low for Pets.

**Note on Pets.** We collected only 68 Pets replies (other topics had 139–226), and 53 are left after cleaning. Only 3 of them are toxic, so one tweet changes the % by about 2 points, and section 7 shows that 2 of the 3 are jokes. So "Pets is more toxic than we guessed" is not a reliable result.

## 5. Question 2: What kind of toxicity?

For each toxic tweet, we take the type with the highest Detoxify score: obscene, insult, threat, identity attack or sexual. We skip `toxicity` and `severe_toxicity`, because they say *how* toxic, not *what kind*. Posts and replies are combined to get more tweets. Topics with a small n are not reliable.

In [ ]:
TYPE_COLS = ["obscene", "insult", "threat", "identity_attack", "sexual_explicit"]

# for each toxic tweet, the type with the highest Detoxify score
tox = scored[scored["detox_toxic"] == 1].copy()
type_scores = tox[["detox_" + c for c in TYPE_COLS]]
tox["tox_type"] = type_scores.idxmax(axis=1).str.replace("detox_", "")
tox[["topic", "tox_type", "text_clean"]].head(3)

In [ ]:
types = pd.crosstab(tox["topic"], tox["tox_type"]).reindex(index=TOPIC_ORDER, columns=TYPE_COLS).fillna(0)
n_toxic = types.sum(axis=1).astype(int)
types_pct = types.div(n_toxic, axis=0) * 100
TYPE_COLOR = {"obscene": "#e67e22", "insult": "#c0392b", "threat": "#8e44ad",
              "identity_attack": "#2c3e50", "sexual_explicit": "#16a085"}
SMALL = 10   # topics with fewer toxic tweets than this are drawn lighter (not reliable)

fig, ax = plt.subplots(figsize=(9, 3.8))
for y, t in enumerate(TOPIC_ORDER):
    alpha = 0.35 if n_toxic[t] < SMALL else 1.0
    left = 0
    for c in TYPE_COLS:
        v = types_pct.loc[t, c]
        ax.barh(y, v, left=left, height=0.7, color=TYPE_COLOR[c], alpha=alpha, edgecolor="white",
                label=c if y == 0 else None)
        if v >= 12:
            ax.text(left + v / 2, y, f"{v:.0f}%", ha="center", va="center", fontsize=8,
                    color="white" if alpha == 1 else "black")
        left += v
ax.set_yticks(range(6), [f"{TOPIC_NAME[t]} (n={n_toxic[t]})" for t in TOPIC_ORDER])
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("% of toxic tweets (Detoxify, posts and replies)")
ax.set_title(f"Most common type of toxicity per topic (lighter = fewer than {SMALL} toxic tweets)")
ax.legend(frameon=False, bbox_to_anchor=(1, 1), loc="upper left")
plt.tight_layout()
plt.savefig(FIG / "fig4_toxicity_types.png", bbox_inches="tight")
plt.show()

types_table = types_pct.round(0)
types_table["n_toxic"] = n_toxic
types_table.rename(index=TOPIC_NAME)

**Answer to Question 2**

- **Insult is the most common type** in the topics with enough toxic tweets: US Politics 75% (of 75 toxic tweets), K-pop 78% (of 18) and Movies 62% (of 13). People mostly attack each other, not swear in general.
- **Obscene (swearing) is second**, at 21–23% in these three topics.
- **Threat, identity attack and sexual content are rare** (8% or less everywhere).
- **Tech, Pets and Food have only 4–7 toxic tweets each**, so their mix is not reliable. Their top type is often obscene (e.g. 3 of 5 in Pets), and section 7 shows that many of these are friendly swearing, not attacks.

## 6. Question 3: How accurate is each method?

Sections 4 and 5 compare the methods with **each other** and with **our guess**, but not with the truth. Reading the top tweets (section 7) finds false alarms, but it can never find toxic tweets that a method **missed**. So we labelled a sample of replies by hand.

**The sample (204 replies).** We split all 817 replies into two groups:

```
817 replies
 ├─ flagged by at least one method: 102  → take all 102     → precision
 └─ not flagged by any method:      715  → take 17 per topic → recall
                                           (17 × 6 = 102)
                       total to label = 204
```

- **Flagged** = toxic for the word list **or** Detoxify **or** RoBERTa (at our normal rule). We take **all** of them, so precision is exact, not a guess.
- **Not flagged:** we take 17 random replies from each topic (same number as the flagged group, and every topic gets the same chance). This is where we look for toxic replies that all three methods missed. Each sampled reply stands for all the un-flagged replies of its topic (its **weight**, e.g. 116 / 17 ≈ 6.8 for US Politics), so recall is estimated for the whole dataset.

**How we labelled.** Two members labelled every reply **alone**, without seeing the topic, the scores or the other person's labels (`data/labelling/labeller_A.csv` and `labeller_B.csv`, rules in `INSTRUCTIONS.txt`). 1 = toxic (rude, disrespectful or unreasonable, likely to make someone leave the conversation), 0 = not toxic. A third member decided the rows where they disagreed.

The sample was made with `make_label_sample.py`. The next cell makes it again and checks that it is the same.

In [ ]:
LABEL_DIR = DATA / "labelling"
N_UNFLAGGED_PER_TOPIC = 17

rep = scored[scored["type"] == "reply"].copy()
rep["flagged"] = (rep["bw_toxic"] == 1) | (rep["detox_toxic"] == 1) | (rep["roberta_toxic"] == 1)

# same steps and seed as make_label_sample.py
parts = []
for topic, g in rep[~rep["flagged"]].groupby("topic"):
    parts.append(g.sample(n=N_UNFLAGGED_PER_TOPIC, random_state=SEED))
unflagged = pd.concat(parts)
sample_ids = set(rep.loc[rep["flagged"], "tweet_id"]) | set(unflagged["tweet_id"])

key = pd.read_csv(LABEL_DIR / "key.csv", dtype={"tweet_id": str}, encoding="utf-8-sig")
assert set(key["tweet_id"]) == sample_ids, "key.csv does not match the sample - was scored.csv changed?"
print(f"key.csv matches: {len(key)} replies ({rep['flagged'].sum()} flagged + {len(unflagged)} un-flagged)")

counts = pd.DataFrame(index=TOPIC_ORDER)
counts["replies"] = rep.groupby("topic").size()
counts["flagged"] = rep.groupby("topic")["flagged"].sum()
counts["unflagged"] = counts["replies"] - counts["flagged"]
counts["sampled"] = counts["flagged"] + N_UNFLAGGED_PER_TOPIC
counts = counts.rename(index=TOPIC_NAME)
counts.index.name = "topic"
counts.loc["Total"] = counts.sum()
counts

### 6.1 Do the two labellers agree?

We read both label files and measure their agreement with **Cohen's kappa** (0 = no better than chance, 1 = perfect). If the two people often disagree, "toxic" is not a clear label and the scores below are less certain.

The rows where they disagree are saved to `data/labelling/disagreements.csv`. The third member fills in the `final` column (1 or 0) and saves it as **CSV UTF-8**.

In [ ]:
def read_labels(name):
    # returns None if the file is not finished yet
    path = LABEL_DIR / f"labeller_{name}.csv"
    try:
        sheet = pd.read_csv(path, dtype=str, encoding="utf-8-sig")
    except UnicodeDecodeError:
        print(f"{path.name}: can't read it - in Excel, save it again as 'CSV UTF-8'.")
        return None
    label = pd.to_numeric(sheet["label"].str.strip(), errors="coerce")
    bad = sheet.loc[~label.isin([0, 1]), "sample_id"].tolist()
    if len(bad) > 0:
        print(f"{path.name}: {len(bad)} rows have no label or not 0/1 (first: {bad[:5]})")
        return None
    sheet["label"] = label.astype(int)
    return sheet.set_index("sample_id")


lab_a = read_labels("A")
lab_b = read_labels("B")
labels_ready = lab_a is not None and lab_b is not None

if labels_ready:
    both = lab_a[["label", "notes"]].join(lab_b[["label", "notes"]], lsuffix="_A", rsuffix="_B")
    agree = both["label_A"] == both["label_B"]
    kappa = cohen_kappa_score(both["label_A"], both["label_B"])
    print(f"Agree on {agree.sum()} of {len(both)} replies ({agree.mean():.0%}), Cohen's kappa = {kappa:.2f}")
    print(f"Toxic: A says {both['label_A'].sum()}, B says {both['label_B'].sum()}")

    # save the disagreements for the third member (never overwrite their file)
    DISAGREE = LABEL_DIR / "disagreements.csv"
    if not agree.all() and not DISAGREE.exists():
        dis = lab_a.loc[~agree, ["parent_text", "reply_text"]]
        dis = dis.join(both.loc[~agree, ["label_A", "label_B", "notes_A", "notes_B"]])
        dis["final"] = ""
        dis.to_csv(DISAGREE, encoding="utf-8-sig")
        print(f"Saved {(~agree).sum()} disagreements to {DISAGREE} - the third member fills in 'final'.")
else:
    print("Waiting for both labellers to finish.")

### 6.2 Which method is right?

Final label = the label both labellers gave, or the third member's decision when they disagreed. Then for each method:

- **Precision:** of the replies the method calls toxic, how many are really toxic?
- **Recall:** of all the really toxic replies, how many does the method find?
- **F1:** one number that balances the two. **This is our main number.**
- **Accuracy:** % of all replies the method gets right. We show it, but it is misleading here: only about 9% of replies are toxic, so a method that never says toxic already gets about 91%.

The counts (TP, FP, FN) are **weighted**: each un-flagged reply in the sample counts for all the un-flagged replies of its topic, so they are estimates for all 817 replies. A flagged reply counts as 1.

In [ ]:
final = None
if labels_ready:
    final = both["label_A"].where(agree)   # empty where A and B disagree
    if not agree.all():
        dis = pd.read_csv(LABEL_DIR / "disagreements.csv", dtype=str, encoding="utf-8-sig")
        dis = dis.set_index("sample_id")
        decided = pd.to_numeric(dis["final"].str.strip(), errors="coerce")
        if decided.isin([0, 1]).all():
            final = final.fillna(decided)
        else:
            n_missing = (~decided.isin([0, 1])).sum()
            print(f"disagreements.csv: {n_missing} rows still need a 'final' label (1 or 0).")
            final = None

if final is None:
    print("Final labels not ready yet.")
else:
    ev = key.set_index("sample_id").join(final.astype(int).rename("label"))
    w = ev["weight"]
    y = ev["label"]
    est_toxic = (w * y).sum()
    n_flagged_toxic = y[ev["stratum"] == "flagged"].sum()
    n_unflagged_toxic = y[ev["stratum"] == "unflagged"].sum()
    print(f"Hand labels: {y.sum()} of {len(ev)} sampled replies are toxic "
          f"({n_flagged_toxic} of 102 flagged, {n_unflagged_toxic} of 102 un-flagged).")
    print(f"Estimated toxic replies in all 817: {est_toxic:.0f} ({est_toxic / len(rep):.1%})")

    def scores(pred):
        # weighted counts, so they are estimates for all 817 replies
        tp = (w * (pred & (y == 1))).sum()
        fp = (w * (pred & (y == 0))).sum()
        fn = (w * (~pred & (y == 1))).sum()
        p = tp / (tp + fp) if tp + fp > 0 else float("nan")
        r = tp / (tp + fn) if tp + fn > 0 else float("nan")
        f1 = round(2 * p * r / (p + r), 2) if p + r > 0 else 0.0
        acc = 1 - (fp + fn) / w.sum()   # the weights add up to all 817 replies
        return {"says toxic": round(tp + fp), "TP": round(tp, 1), "FP": round(fp, 1), "FN": round(fn, 1),
                "precision": round(p, 2), "recall": round(r, 2), "F1": f1, "accuracy": round(acc, 3)}

    rule = {"Word list": ev["flag_wordlist"],
            "Detoxify": ev["detox_toxicity"] >= THRESHOLD,
            "RoBERTa": ev["roberta_offensive"] >= THRESHOLD}
    lab_results = pd.DataFrame({name: scores(pred) for name, pred in rule.items()}).T
    lab_results["says toxic"] = lab_results["says toxic"].astype(int)
    display(lab_results)

In [ ]:
if final is not None:
    METRICS = ["precision", "recall", "F1"]
    METHOD_COLOR = {"Detoxify": "#c0392b", "RoBERTa": "#2980b9", "Word list": "#7f8c8d"}
    fig, ax = plt.subplots(figsize=(8, 3.6))
    for i, name in enumerate(METHOD_COLOR):
        x = np.arange(3) + (i - 1) * 0.27
        vals = lab_results.loc[name, METRICS].astype(float)
        ax.bar(x, vals, 0.27, label=name, color=METHOD_COLOR[name])
        for xi, v in zip(x, vals):
            ax.text(xi, v + 0.02, f"{v:.2f}", ha="center", fontsize=8)
    ax.set_xticks(np.arange(3), ["Precision", "Recall", "F1"])
    ax.set_ylim(0, 1.1)
    ax.set_title("How accurate is each method? (hand labels, estimated for all 817 replies)")
    ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.12))
    plt.tight_layout()
    plt.savefig(FIG / "fig5_method_scores.png", bbox_inches="tight")
    plt.show()

### 6.3 Was 0.5 a good threshold?

We used 0.5, the usual default (section 4.2). Now we can check which threshold is actually **right** most often (highest F1). The weights also work here, because the sample was chosen at 0.5 but the un-flagged weights stand for every un-flagged reply.

**How often is a flagged reply really toxic, by how many methods flagged it?** If all three methods agree, we expect the reply to be toxic almost every time.

In [ ]:
if final is not None:
    # precision / recall / F1 at each threshold
    rows = []
    for name, col in SCORES.items():
        for th in THRESHOLDS:
            row = scores(ev[col] >= th)
            row["method"] = name
            row["threshold"] = th
            rows.append(row)
    th_table = pd.DataFrame(rows).set_index(["method", "threshold"])
    display(th_table[["says toxic", "precision", "recall", "F1"]])

    # how many methods flagged it vs how often it is really toxic
    fl = ev[ev["stratum"] == "flagged"].copy()
    fl["n_methods"] = fl["flag_wordlist"].astype(int) + fl["flag_detoxify"].astype(int) + fl["flag_roberta"].astype(int)
    agree_table = fl.groupby("n_methods")["label"].agg(["size", "sum"])
    agree_table.columns = ["replies", "really_toxic"]
    agree_table["pct"] = (agree_table["really_toxic"] / agree_table["replies"] * 100).round(0)
    agree_table.index.name = "methods that flagged it"
    display(agree_table)

**Answer to Question 3**

- **The two labellers agree well:** 183 of 204 replies (90%), kappa 0.76. So "toxic" is a clear enough label to judge the methods with.
- **Detoxify and RoBERTa are about equally good overall** (F1 0.74 and 0.75), but in different ways. **Detoxify is more careful:** when it says toxic it is right 79% of the time, but it finds 69% of the toxic replies. **RoBERTa finds more** (85%) but has more false alarms (right 67% of the time).
- **The word list is the weakest** (F1 0.43). When it finds a bad word it is usually right (precision 0.88), but it finds only 29% of the toxic replies. It knows swear words, not insults like *stupid* or *trash*.
- **Accuracy hides all this:** every method gets 93–96%, because about 91% of replies are not toxic anyway.
- **More methods agreeing = more likely really toxic:** 95% of replies flagged by all 3 methods are toxic, but only 41% of replies flagged by just one.
- **Threshold:** 0.5 is the best of the three for RoBERTa (F1 0.75; at 0.2 it flags 305 replies and most are false alarms). For Detoxify, 0.2 is a little better than 0.5 (F1 0.79 vs. 0.74), because it finds more toxic replies without many more false alarms.

**Limits.** Recall comes from only 17 un-flagged replies per topic. Just 1 of those 102 was labelled toxic (a borderline US Politics reply that the labellers disagreed on), and it counts for about 7 replies, so the recall numbers would change noticeably if that one label changed. The labels are our own judgement.

## 7. A closer look at the tweets

The AI models give **one score for the whole tweet**, so they don't say which word is the problem. To find it, for every toxic tweet (score 0.5 or more) we **remove one word at a time and score the tweet again**. The 2 words whose removal lowers the score most are the tweet's **trigger words**. We skip small words ("the", "to"), @names, links and emoji.

This takes about 8 minutes, so we save the result to `data/trigger_words.csv` and reuse it.

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

TRIGGERS = DATA / "trigger_words.csv"
MODELS = {"Detoxify": (DETOX_ID, "detox_toxic"), "RoBERTa": (ROBERTA_ID, "roberta_toxic")}
SKIP = ENGLISH_STOP_WORDS | {"user", "http", "amp", "rt"}


def norm_word(w):
    # 'Shit.' -> 'shit', and '' for words we skip (stopwords, @user, emoji, it's ...)
    w = w.lower().strip(".,!?;:\"()[]…“”‘’-").lstrip("#")
    if not re.search("[a-z]", w) or "'" in w or "’" in w or w in SKIP:
        return ""
    return w


def find_triggers(name, model_id, tweets, top_n=2):
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSequenceClassification.from_pretrained(model_id)
    model.eval()

    def raw_score(texts):
        # score before sigmoid/softmax
        # Detoxify: column 0 = toxicity, RoBERTa: offensive - non-offensive
        with torch.no_grad():
            enc = tok(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
            logits = model(**enc).logits
        if name == "Detoxify":
            return logits[:, 0].numpy()
        return (logits[:, 1] - logits[:, 0]).numpy()

    rows = []
    start = time.time()
    for tid, text in zip(tweets["tweet_id"], tweets["text_model"]):
        words = text.split()
        # the tweet itself, then the tweet with each word removed
        texts = [text]
        for i in range(len(words)):
            texts.append(" ".join(words[:i] + words[i + 1:]))
        s = raw_score(texts)

        drop = {}
        for w, d in zip(words, s[0] - s[1:]):
            w = norm_word(w)
            if w and d > 0:
                drop[w] = max(d, drop.get(w, 0))
        top = sorted(drop.items(), key=lambda x: -x[1])[:top_n]
        for w, d in top:
            rows.append({"model": name, "tweet_id": tid, "word": w, "drop": round(float(d), 3)})
    print(f"{name}: {len(tweets)} toxic tweets in {time.time() - start:.0f}s")
    return rows


# reuse trigger_words.csv if it was made after the last scoring
trig = None
if TRIGGERS.exists() and TRIGGERS.stat().st_mtime > SCORED.stat().st_mtime:
    trig = pd.read_csv(TRIGGERS, dtype={"tweet_id": str})
    print(f"Loaded saved trigger words. Delete {TRIGGERS} to compute again.")
if trig is None:
    rows = []
    for name, (model_id, col) in MODELS.items():
        rows += find_triggers(name, model_id, scored[scored[col] == 1])
    trig = pd.DataFrame(rows)
    trig.to_csv(TRIGGERS, index=False, encoding="utf-8")

In [ ]:
from html import escape
from IPython.display import HTML

TRIG = trig.groupby(["model", "tweet_id"])["word"].agg(set)


def bold(text, words):
    # escape for HTML and make the trigger words bold
    text = escape(text)
    if not words:
        return text
    words = sorted(words, key=len, reverse=True)
    pat = re.compile(r"\b(" + "|".join(re.escape(w) for w in words) + r")\b", re.I)
    return pat.sub(r"<b>\1</b>", text)


def show_tweets(rows, models=("Detoxify", "RoBERTa")):
    texts = []
    for tid, text in zip(rows["tweet_id"], rows["text_clean"]):
        words = set()
        for m in models:
            words |= TRIG.get((m, tid), set())
        texts.append(bold(text, words))

    t = pd.DataFrame()
    t["topic"] = rows["topic"].map(TOPIC_NAME)
    t["type"] = rows["type"]
    t["Detoxify"] = rows["detox_toxicity"].map("{:.2f}".format)
    t["RoBERTa"] = rows["roberta_offensive"].map("{:.2f}".format)
    t["bad words"] = rows["bw_words"].fillna("").str.replace("|", ", ", regex=False)
    t["text"] = texts
    html = t.to_html(index=False, escape=False).replace("<td>", '<td style="text-align:left">')
    display(HTML(html))

### 7.1 The 15 highest-scoring tweets for each model

**Bold** = trigger words of that model. `bad words` = words found by the bad-word list.

In [ ]:
TOP_N = 15

print(f"Top {TOP_N} by Detoxify")
top = scored.sort_values(["detox_toxicity", "roberta_offensive"], ascending=False).head(TOP_N)
show_tweets(top, models=["Detoxify"])

print(f"Top {TOP_N} by RoBERTa")
top = scored.sort_values(["roberta_offensive", "detox_toxicity"], ascending=False).head(TOP_N)
show_tweets(top, models=["RoBERTa"])

**What we see**

- **Almost all of the top tweets are US Politics:** 14 of Detoxify's top 15 and 13 of RoBERTa's. The two lists share 9 tweets.
- **They are personal attacks.** The bold trigger words show that the models react to insults (*stupid, pathetic, dumbest, idiot, morons*) as much as to swear words. 6 of Detoxify's top 15 have **no word from the bad-word list**, so the list would miss them.
- **Some trigger words are not insults:** *Democrats*, *vote*, *deport*. So the models partly react to political words themselves, which may make US Politics look slightly more toxic.
- **High score does not always mean hostile.** "girl u crazy, crazy ass a bottle of dry water ... cutie" is a joke (a meme using "ass a" for "as a"), but both models give it 0.93–1.00 because "ass" appears 5 times.

### 7.2 The most toxic tweets in each topic

To see **why** each topic got its level in Question 1, we show the 10 tweets with the highest Detoxify score in every topic (posts and replies together). **Bold** = trigger words of either model.

In [ ]:
TOP_TOPIC = 10
reply_levels = to_levels(reply.groupby("topic")["detox_toxic"].mean().reindex(TOPIC_ORDER))
for t in TOPIC_ORDER:
    g = scored[scored["topic"] == t]
    print(f"{TOPIC_NAME[t]}: our guess {GUESS[t]}, data {reply_levels[t]} (replies, Detoxify), "
          f"{g['detox_toxic'].sum()} of {len(g)} tweets toxic")
    top = g.sort_values(["detox_toxicity", "roberta_offensive"], ascending=False).head(TOP_TOPIC)
    show_tweets(top)

**What we see**

- **US Politics and K-pop:** the top 10 are real attacks on people or fans (*stupid cunt*, *#BLACKPINK fans are trash*). One K-pop mistake: the song title "JENNIE — Damn Right" gets Detoxify 0.84 (RoBERTa 0.09).
- **Movies is mixed:** real insults (*colossal incompetent retard*, *scumbags*) next to false alarms: "OMG THEY'RE BABIESSSS" (0.84), a film called *Pathetic Fallacy* (0.76) and a Richard Pryor quote about swearing.
- **Tech has very few toxic tweets.** Only 3 posts score above 0.9. Below that the Detoxify scores drop fast, which is why Tech came out Low. But RoBERTa finds sexist replies that Detoxify misses, e.g. "Girls women empowerment freedom to become hoe and criminal 🤡" (Detoxify 0.07, RoBERTa 0.78). So Detoxify may count Tech as less toxic than it is.
- **Pets (guessed Low, came out Mid):** its Mid level comes from just 3 toxic replies. Only 1 is a real attack: "Get over it cunt.", written to someone whose dog had died. The other 2, "Such meanies!" and "Slap the dog", are jokes under a playful cat post ("It's so unfair....👇😉❤️"). The 2 toxic Pets posts are also pet talk: a kitten who is "a little shit sometimes" and a "tiny pussy cat".
- **Food (Low):** the same pattern. Only "Couldn't pay me to eat that processed shit. Recipe for morons by morons." is an attack. The others are friendly swearing ("Damn these look so good!", 0.98), double meanings ("I'm a slut for dominos pizza") or violent words in a harmless context ("kill two characters", "chop this one").

**Why calm topics still get toxic tweets:** in Pets and Food, the models are fooled by bad words used in a friendly way, words with a double meaning, and violent words in a harmless context. They score the words, not what the person means. The bad-word list has the same problem with *shit*, *pussy* and *slut*. So for calm topics, a few % "toxic" mostly means false alarms, not people attacking each other.